# 01 — Data Exploration

**What this notebook is for:** understanding what actually arrives from each data
source before any modelling decision is made.

AFYA-PREDICT fuses six or more independent streams onto one `district x
epidemiological week` grid. Before trusting a forecast built on that grid, it is
worth answering three questions directly from the data:

1. **What does each adapter actually produce**, and on what cadence?
2. **How complete is it**, and where are the gaps concentrated?
3. **Do the fused series behave the way the epidemiology says they should** —
   does rainfall lead malaria, does the dry season raise particulates?

> **A note on synthetic data.** With no credentials configured, every adapter
> falls back to a deterministic synthetic climatology. That is a deliberate
> design choice (see `src/data_ingestion/base_adapter.py`) so the whole pipeline
> is runnable on day one. Numbers produced here are therefore *demonstrative,
> not operational*. The `mode` column below tells you which is which, and it is
> the first thing to check on a real deployment.

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

## 1. The spatial grid

Every source is resampled onto this council-level grid.

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

## 2. What can each adapter provide?

`describe_sources()` reports, for every registered adapter, which variables it
produces, how often upstream publishes, and — critically — whether credentials
for a **live** fetch are present on this machine.

In [ ]:
from src.data_ingestion.registry import describe_sources, available_sources

sources = pd.DataFrame(describe_sources(region=REGION))
sources["variables"] = sources["variables"].apply(lambda v: ", ".join(v))
print(f"{len(sources)} adapters registered; "
      f"{int(sources['configured'].sum())} configured for live fetch\n")
sources[["source", "configured", "optional", "update_frequency_days",
         "native_resolution", "variables"]]

Note `search_trends` is the only adapter flagged `optional`. That is
enforced, not cosmetic: Tanzanian smartphone penetration is around 35% against
~87% feature-phone ownership, so search interest systematically under-observes
the rural districts carrying most of the burden. The platform refuses to let it
act as a primary predictor (shortcoming #14).

## 3. Ingest a fused panel

`ingest()` runs each adapter's `fetch -> validate -> normalize` pipeline and hands
the results to the normaliser, which aligns everything onto the district x week
grid and records a quality score per cell.

In [ ]:
from src.data_ingestion.normalizer import ingest

SOURCES = ["chirps", "era5", "modis", "sentinel5p", "dhis2",
           "cdr_mobility", "population_density", "wash_indicators",
           "livestock_disease"]
START_WEEK, END_WEEK = "2021-W01", "2024-W52"

panel = ingest(SOURCES, START_WEEK, END_WEEK, region=REGION)

summary = panel.summary()
print(f"districts     : {summary['districts']}")
print(f"weeks         : {summary['weeks']}")
print(f"variables     : {summary['variables']}")
print(f"sources       : {', '.join(summary['sources'])}")
print(f"completeness  : {summary['completeness']:.1%}")
print(f"mean quality  : {summary['mean_quality']:.2f}")
print("\nfetch mode per source (live / cache / synthetic):")
for source, mode in sorted(summary["modes"].items()):
    print(f"  {source:22} {mode}")

### Read the quality flags before the data

Flags are not warnings to skim past — they propagate into the confidence
intervals and can stamp an alert `LOW DATA CONFIDENCE`. The rule the ingestion
layer follows is **flag, never silently drop or zero-fill**.

In [ ]:
flags = pd.DataFrame([f.model_dump() for f in panel.flags])
if flags.empty:
    print("no quality flags raised")
else:
    print(f"{len(flags)} flag(s) raised\n")
    display(flags.groupby(["severity", "code"]).size().rename("count").reset_index())
    print("\nMost severe first:\n")
    for _, row in flags[flags.severity.isin(["error", "warning"])].head(8).iterrows():
        print(f"  [{row.severity.upper():7}] {row.source or 'pipeline'}: {row.message}")

## 4. Completeness — where are the gaps?

DHIS2 nationally reports around 93.9% completeness, but that average hides the
districts that matter. Roughly 70% of deaths in Tanzania occur outside health
facilities, so surveillance **undercounts true burden** — which is why a missing
week is kept as `NaN` with `quality = 0` rather than being read as "zero cases".

In [ ]:
values = panel.values()
quality = panel.quality()

completeness = pd.DataFrame({
    "completeness": values.notna().mean(),
    "mean_quality": quality.mean(),
    "source": pd.Series(panel.sources),
}).sort_values("completeness")

print("Least complete variables:")
display(completeness.head(10).style.format({"completeness": "{:.1%}", "mean_quality": "{:.2f}"})
        if hasattr(pd.DataFrame, "style") else completeness.head(10))

case_cols = [c for c in values.columns if c.startswith("cases_")]
print("\nSurveillance completeness by district (cases_malaria):")
by_district = (values["cases_malaria"].groupby(level="district").apply(lambda s: s.notna().mean())
               if "cases_malaria" in values else pd.Series(dtype=float))
by_district.sort_values().round(3)

## 5. Do the series behave epidemiologically?

This is the sanity check that matters most. Rainfall should be seasonal with a
bimodal signature in the north and unimodal in the south; particulates should
peak in the dry season; case counts should lag their climate drivers.

In [ ]:
drivers = ["rainfall_mm", "temperature_c", "humidity_pct", "ndvi", "pm25_ug_m3"]
available = [d for d in drivers if d in values.columns]

national = values[available].groupby(level="week").mean()
national = national.reindex(sorted(national.index))

if HAS_PLT:
    fig, axes = plt.subplots(len(available), 1, figsize=(12, 2.1 * len(available)), sharex=True)
    for ax, column in zip(np.atleast_1d(axes), available):
        ax.plot(range(len(national)), national[column], lw=1.2)
        ax.set_ylabel(column, fontsize=9)
    axes[-1].set_xlabel("week index")
    fig.suptitle("Driver series, mean across study districts", y=0.995)
    plt.tight_layout()
    plt.show()
else:
    display(national[available].describe().T.round(2))

### Seasonal shape by week-of-year

Collapsing across years exposes the transmission season directly.

In [ ]:
woy = pd.Index([int(w.split("-W")[1]) for w in national.index], name="week_of_year")
seasonal = national.groupby(woy).mean()

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(12, 4))
    for column in available:
        normalised = (seasonal[column] - seasonal[column].min()) / (
            seasonal[column].max() - seasonal[column].min())
        ax.plot(seasonal.index, normalised, label=column, lw=1.6)
    ax.axvspan(9, 21, alpha=0.10, color="tab:blue", label="masika (long rains)")
    ax.axvspan(40, 52, alpha=0.10, color="tab:cyan", label="vuli (short rains)")
    ax.set_xlabel("ISO week of year"); ax.set_ylabel("min-max normalised")
    ax.legend(ncol=3, fontsize=8); ax.set_title("Seasonal shape of each driver")
    plt.show()
else:
    display(seasonal.round(2))

## 6. Cross-correlation: does rainfall lead malaria?

This is the whole premise of the platform. If climate drivers did **not** lead
case counts, there would be no forecast horizon to exploit and the system would
be a nowcasting tool at best.

The peak of this curve is an early estimate of the transmission lag. The modelling
layer does not take it on trust — `fit_optimal_lags` refits it per district
(critical rule #3) — but if the national curve is flat, something is wrong upstream.

In [ ]:
def cross_correlation(driver: pd.Series, target: pd.Series, max_lag: int = 20):
    """Spearman correlation of a driver against a target at each lead time."""
    rows = []
    for lag in range(max_lag + 1):
        pair = pd.concat([driver.shift(lag), target], axis=1).dropna()
        rho = pair.iloc[:, 0].corr(pair.iloc[:, 1], method="spearman") if len(pair) > 12 else np.nan
        rows.append({"lag_weeks": lag, "spearman": rho})
    return pd.DataFrame(rows).set_index("lag_weeks")

district = "Sengerema"
local = values.xs(district, level="district").sort_index()

curves = {}
for driver, target in [("rainfall_mm", "cases_malaria"),
                       ("temperature_c", "cases_malaria"),
                       ("rainfall_mm", "cases_cholera")]:
    if driver in local and target in local:
        curves[f"{driver} -> {target}"] = cross_correlation(local[driver], local[target])["spearman"]

curve_frame = pd.DataFrame(curves)
best = curve_frame.abs().idxmax()
print(f"District: {district}\n")
print("Lag with the strongest association:")
for name, lag in best.items():
    print(f"  {name:34} {lag:>2} weeks  (rho = {curve_frame.loc[lag, name]:+.3f})")

if HAS_PLT:
    ax = curve_frame.plot(figsize=(11, 4), marker="o", ms=3)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("driver lead time (weeks)"); ax.set_ylabel("Spearman rho")
    ax.set_title(f"Cross-correlation, {district}")
    plt.show()
else:
    display(curve_frame.round(3))

## 7. Spatial structure

Two districts can share a climate and behave completely differently. The gravity
matrix below is the fallback used when negotiated CDR mobility data is not
available (critical rule #8) — the platform never depends on a telco agreement
to produce spatial predictions.

In [ ]:
from src.core.geo import distance_matrix, gravity_matrix

distances = distance_matrix(REGION).round(0)
flows = gravity_matrix(REGION).round(3)

print("Great-circle distance (km):")
display(distances)
print("\nGravity-model flow, row = origin, column = destination (rows sum to 1):")
display(flows)
print("\nStrongest corridors:")
pairs = (flows.stack().rename("flow").reset_index()
         .rename(columns={"level_0": "origin", "level_1": "destination"}))
pairs[pairs.origin != pairs.destination].nlargest(6, "flow").reset_index(drop=True)

## 8. What to check on a real deployment

Before believing anything downstream of this notebook:

| Check | Where | Why it matters |
|---|---|---|
| `mode` is `live` for ≥3 sources | section 3 | Critical rule #1 — no single-source dependence |
| No `error`-severity flags | section 3 | Errors mean a feed is broken, not merely noisy |
| Surveillance completeness > ~0.8 per district | section 4 | Below this the target is too sparse to fit locally |
| Cross-correlation peaks at a plausible lag | section 6 | A flat curve means no exploitable lead time |
| Districts differ in their peak lag | notebook 02 | Confirms per-district fitting is necessary |

Next: **`02_lag_analysis.ipynb`** takes the lag question seriously and shows why
a single national coefficient would be wrong.